# Overlay G/T sul PER — solo sensori

Crea results/per_overlay prima di salvare. n_run=1, n_proc=1, force_overwrite=True.


In [7]:
import os
if not os.path.exists("Drosophila_brain_model/model.py"):
    !git clone --depth 1 https://github.com/philshiu/Drosophila_brain_model.git
    !pip install -q brian2 pandas pyarrow
%cd Drosophila_brain_model


Cloning into 'Drosophila_brain_model'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 23 (delta 1), reused 17 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 181.74 MiB | 35.48 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/Drosophila_brain_model/Drosophila_brain_model


In [8]:
from model import run_exp
from model import default_params as params
import utils as utl
import pandas as pd
from brian2 import Hz
config = {"path_res": "./results/per_overlay", "path_comp": "./Completeness_783.csv", "path_con": "./Connectivity_783.parquet", "n_proc": 1}
os.makedirs(config["path_res"], exist_ok=True)
params["n_run"] = 1
params["r_poi"] = 100 * Hz
_comp = set(pd.read_csv(config["path_comp"]).iloc[:,0].astype("int64"))
def keep(name, lst):
    ok = [i for i in lst if i in _comp]
    print(name, "tenuti", len(ok), "/", len(lst), "scartati", [i for i in lst if i not in _comp])
    return ok


In [9]:
neu_sugar = keep("GRN", [720575940624963786,720575940630233916,720575940637568838,720575940638202345,720575940617000768,720575940630797113,720575940632889389,720575940621754367,720575940621502051,720575940640649691,720575940639332736,720575940616885538,720575940639198653,720575940617937543,720575940632425919,720575940633143833,720575940612670570,720575940628853239,720575940629176663,720575940611875570])
MN9_L = 720575940618238523
MN9_R = 720575940660219265
DH44 = keep("DH44", [720575940616153371,720575940618385718,720575940626619398,720575940622872870,720575940623011845,720575940635861146])
CRZ = keep("CRZ", [720575940609268316,720575940610895474,720575940614254534,720575940619845552,720575940620332141,720575940642666440])
HUGIN = keep("HUGIN", [720575940637851069,720575940638394446,720575940627293201,720575940643847022])


GRN tenuti 20 / 20 scartati []
DH44 tenuti 6 / 6 scartati []
CRZ tenuti 6 / 6 scartati []
HUGIN tenuti 4 / 4 scartati []


In [10]:
def hill(x, K, n=2.0):
    if x <= 0 or K <= 0:
        return 0.0
    return (x ** n) / (K ** n + x ** n)
NODES = {
    "DH44": dict(ids=DH44, rmax=50.0, wG=1.0, KG=7.0, nG=2.0, wT=0.5, KT=20.0, nT=2.0),
    "CRZ": dict(ids=CRZ, rmax=50.0, wG=1.0, KG=4.0, nG=2.0, wT=0.3, KT=20.0, nT=2.0),
    "HUGIN": dict(ids=HUGIN, rmax=50.0, wG=1.0, KG=7.0, nG=2.0, wT=0.0, KT=1.0, nT=2.0),
}
def drive_hz(node, G, T):
    p = NODES[node]
    return p["rmax"] * (p["wG"] * hill(G, p["KG"], p["nG"]) + p["wT"] * hill(T, p["KT"], p["nT"]))
def overlay_ids_and_rate(G, T):
    rates, ids = [], []
    for name, p in NODES.items():
        r = drive_hz(name, G, T)
        rates.append((name, r, len(p["ids"])))
        ids.extend(p["ids"])
    r2 = sum(r for _, r, _ in rates) / len(rates)
    return ids, r2, rates


In [ ]:
T_FIX = 30.0
conditions = [("G2_T30", 2.0, T_FIX), ("G20_T30", 20.0, T_FIX)]
for name, G, T in conditions:
    ids2, r2, detail = overlay_ids_and_rate(G, T)
    print(f"\n{name}: G={G} mM T={T} mM r_poi2={r2:.1f} Hz")
    for n, r, k in detail:
        print(f"  {n}: {r:.1f} Hz ({k} cellule)")
    params["r_poi2"] = r2 * Hz
    run_exp(exp_name=name, neu_exc=neu_sugar, neu_exc2=ids2, params=params, force_overwrite=True, **config)



G2_T30: G=2.0 mM T=30.0 mM r_poi2=15.1 Hz
  DH44: 21.1 Hz (6 cellule)
  CRZ: 20.4 Hz (6 cellule)
  HUGIN: 3.8 Hz (4 cellule)
>>> Experiment:     G2_T30
    Output file:    results/per_overlay/G2_T30.parquet
    Excited neurons: 36
    Elapsed time:   166 s

G20_T30: G=20.0 mM T=30.0 mM r_poi2=55.0 Hz
  DH44: 61.9 Hz (6 cellule)
  CRZ: 58.5 Hz (6 cellule)
  HUGIN: 44.5 Hz (4 cellule)
>>> Experiment:     G20_T30
    Output file:    results/per_overlay/G20_T30.parquet
    Excited neurons: 36


In [ ]:
print("=== INCOLLA QUESTI NUMERI ===")
print("(controllo overlay OFF: MN9_L 44.67, MN9_R 60.67 Hz)")
for name, G, T in conditions:
    df_spike = utl.load_exps([f"./results/per_overlay/{name}.parquet"])
    df_rate, df_std = utl.get_rate(df_spike, t_run=params["t_run"], n_run=params["n_run"])
    print(f"\n{name} G={G} T={T}")
    for lab, nid in [("MN9_L", MN9_L), ("MN9_R", MN9_R)]:
        if nid in df_rate.index:
            mu = float(df_rate.loc[nid].iloc[0])
            sd = float(df_std.loc[nid].iloc[0]) if nid in df_std.index else float("nan")
            print(f"  {lab} {nid}: {mu:.2f} +/- {sd:.2f} Hz")
        else:
            print(f"  {lab} {nid}: NESSUNO SPIKE")


In [ ]:
import os
import numpy as np
import pandas as pd

# --- GFS IDs (Codex FAFB v783) ---
GF_L = 720575940622838154
GF_R = 720575940632499757

ann = pd.read_csv(
    "https://raw.githubusercontent.com/flyconnectome/flywire_annotations/main/supplemental_files/Supplemental_file1_neuron_annotations.tsv",
    sep="\t",
    usecols=["root_id", "cell_type", "side"],
)
LC4 = keep("LC4", ann.loc[ann.cell_type == "LC4", "root_id"].astype("int64").tolist())
LPLC2 = keep("LPLC2", ann.loc[ann.cell_type == "LPLC2", "root_id"].astype("int64").tolist())
loom = LC4 + LPLC2
print("looming", len(loom), "GF", GF_L, GF_R)

# --- grafo Shiu v783 già sul disco ---
df_comp = pd.read_csv(config["path_comp"], index_col=0)
df_con = pd.read_parquet(config["path_con"])

fly2i = {int(i): k for k, i in enumerate(df_comp.index.astype("int64"))}

def to_idx(ids):
    return {fly2i[int(i)] for i in ids if int(i) in fly2i}

src = {
    "DH44": to_idx(DH44),
    "CRZ": to_idx(CRZ),
    "HUGIN": to_idx(HUGIN),
}
tgt_gf = to_idx([GF_L, GF_R])
tgt_loom = to_idx(loom)
print("indici trovati", {k: len(v) for k, v in src.items()}, "GF", len(tgt_gf), "loom", len(tgt_loom))

pre = df_con["Presynaptic_Index"].to_numpy()
post = df_con["Postsynaptic_Index"].to_numpy()
w = df_con["Excitatory x Connectivity"].to_numpy()

print("=== 1 HOP overlay -> GF / LC4+LPLC2 ===")
for name, S in src.items():
    m_gf = np.isin(pre, list(S)) & np.isin(post, list(tgt_gf))
    m_lo = np.isin(pre, list(S)) & np.isin(post, list(tgt_loom))
    print(f"{name:6} -> GF:      n_edge={int(m_gf.sum()):4d}  peso={w[m_gf].sum():8.1f}")
    print(f"{name:6} -> looming: n_edge={int(m_lo.sum()):4d}  peso={w[m_lo].sum():8.1f}")

print("=== 2 HOP overlay -> X -> GF ===")
from_ov = np.isin(pre, list(set().union(*src.values())))
mid = set(post[from_ov].tolist()) - tgt_gf
to_gf = np.isin(pre, list(mid)) & np.isin(post, list(tgt_gf))
print(f"edges X->GF: {int(to_gf.sum())}  peso secondo salto={w[to_gf].sum():.1f}")